# Week 2 — The model is just a rule you can read

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Catacombs666/ML_int-/blob/main/notebooks/02_your_first_readable_model.ipynb?flush_cache=true)

You'll:
1. Write a **1-line hand rule** and rank pages with it.
2. Fit a **depth-2 decision tree** and `print` it — the model "learned" a readable if/else. Then compare — where does it beat your rule, and where doesn’t it?
3. See **why you never feed the outcome back in** — that's leakage.

The payoff isn't a high score. It's: *my intuition was rough, the model found the real signal, and I can read exactly what it found.*

**Author: Majhabul Hoque**


## 0. Setup (Colab or local)

In [1]:
import os, sys, subprocess, json
from pathlib import Path
candidates=[Path.cwd(),*Path.cwd().parents,Path('/content/ML_int-')]
ROOT=next((p for p in candidates if (p/'data/raw/content_refresh_anonymized.csv').exists()),None)
if ROOT is None:
    ROOT=Path('/content/ML_int-') if Path('/content').exists() else Path.cwd()/'ML_int-'
    subprocess.run(['git','clone','https://github.com/Catacombs666/ML_int-.git',str(ROOT)],check=True)
os.chdir(ROOT)
if os.environ.get('CAPSTONE_OFFLINE')!='1':
    subprocess.run([sys.executable,'-m','pip','install','-q','-r','requirements.txt'],check=True)
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
plt.rcParams['svg.fonttype']='none'
df=pd.read_csv('data/raw/content_refresh_anonymized.csv')
df['is_declining_label']=df['trend_direction'].str.lower().eq('down').astype(int)
print('Starter shape:',df.drop(columns=['is_declining_label']).shape,'| Declining proxy rate:',round(df.is_declining_label.mean(),4))
print('Author: Majhabul Hoque | no client records printed')


Starter shape: (30000, 44) | Declining proxy rate: 0.5421
Author: Majhabul Hoque | no client records printed


## 1. A rule you write by hand: `stale x visible`
Intuition: a page worth reviewing is one that is **stale** (not updated in a while) **and** still **visible** (getting impressions). Rank those by how much exposure they have.

In [2]:
stale   = (df["days_since_last_update"] >= 180).astype(int)
visible = (df["impressions_90d"] >= 500).astype(int)
df["hand_rule_score"] = stale * visible * df["impressions_90d"]

top10 = df.sort_values("hand_rule_score", ascending=False).head(10)
print('Top-10 hand-rule aggregate:',{'pages':len(top10),'median_impressions':float(top10.impressions_90d.median()),'median_update_age_days':float(top10.days_since_last_update.median())})


Top-10 hand-rule aggregate: {'pages': 10, 'median_impressions': 7685.0, 'median_update_age_days': 194.0}


We need a way to score any ranking. **Precision@K** = of the top K pages a ranking flags, what fraction are actually declining.

In [3]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(labels)[order[:k]]
    return topk.mean()

y = df["is_declining_label"].values
for k in (20, 50):
    print(f"Hand rule  Precision@{k}: {precision_at_k(df['hand_rule_score'], y, k):.3f}")

Hand rule  Precision@20: 0.950
Hand rule  Precision@50: 0.660


## 2. Let a model learn the rule — then read it
A **depth-2 decision tree** can only ask 3 yes/no questions. That constraint is the point: whatever it learns, you can read.

We give it a few **pre-decision** signals — never product flags.

In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text

features = ["content_age_days", "days_since_last_update", "impressions_90d",
            "avg_position", "ctr", "word_count"]
X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42)
tree.fit(X, y)

print(export_text(tree, feature_names=features))

|--- impressions_90d <= 5.50
|   |--- avg_position <= 0.75
|   |   |--- class: 0
|   |--- avg_position >  0.75
|   |   |--- class: 0
|--- impressions_90d >  5.50
|   |--- content_age_days <= 312.50
|   |   |--- class: 1
|   |--- content_age_days >  312.50
|   |   |--- class: 0



That printout **is** the model — a human-readable if/else. Now rank pages by the tree's probability and score it the same way.

In [5]:
tree_score = tree.predict_proba(X)[:, 1]
for k in (20, 50):
    hr = precision_at_k(df["hand_rule_score"], y, k)
    tr = precision_at_k(tree_score, y, k)
    print(f"Precision@{k}:  hand rule {hr:.3f}   vs   tree {tr:.3f}")

Precision@20:  hand rule 0.950   vs   tree 0.650
Precision@50:  hand rule 0.660   vs   tree 0.640


Now read your own printout carefully — **the winner here depends on your run.** A depth-2 tree can only give four different scores (one per leaf), so the "top 50" is mostly one big block of tied pages, and different library versions break those ties differently. On some stacks the tree wins at Precision@50; on others the hand rule holds both. **Both results are real.** The stable lesson: a sharp human rule can be excellent at the very top of the list; a model's advantage — when it shows up — appears deeper, where simple rules run out of signal; and any comparison built on heavily tied scores is fragile. Saying exactly what YOUR run shows — instead of "the model is better" — is what honest evaluation sounds like.

## 3. Why you can't feed the outcome back in
Your label is `trend_direction == "down"`, and `trend_pct` is the exact percentage change that bucket is computed from — so it **is** the answer in disguise. Watch what happens if you feed it in as a feature:

In [6]:
X_leaky = df[features + ["trend_pct"]].replace([np.inf, -np.inf], np.nan).fillna(0)
leaky = DecisionTreeClassifier(max_depth=2, class_weight="balanced", random_state=42).fit(X_leaky, y)
print(f"'Leaky' tree Precision@50: {precision_at_k(leaky.predict_proba(X_leaky)[:,1], y, 50):.3f}  <- looks amazing")
print(export_text(leaky, feature_names=features + ["trend_pct"]))

'Leaky' tree Precision@50: 1.000  <- looks amazing
|--- trend_pct <= -20.05
|   |--- word_count <= 212.00
|   |   |--- class: 1
|   |--- word_count >  212.00
|   |   |--- class: 1
|--- trend_pct >  -20.05
|   |--- trend_pct <= -19.95
|   |   |--- class: 0
|   |--- trend_pct >  -19.95
|   |   |--- class: 0



The tree just split on `trend_pct` and nailed the label — because the label is **derived from** `trend_pct`. That's **leakage**: the feature is the answer in disguise, and it teaches you nothing.

That's also why the starter data ships **only observable signals** — the product's own decision flags (health scores, "needs CTR fix", and so on) aren't included, so you can't accidentally train on them. You build from what was knowable *before* the outcome.

> Rule of thumb: if a feature would only be known *because someone already made the decision you're predicting*, it leaks. Leave it out.

## 4. 🔧 Your turn
- Change `max_depth` to 3 or 4 — does Precision@50 improve? Can you still read the tree?
- Swap in different features (drop `impressions_90d`, add `engagement_rate`). What does the tree choose to split on first?
- **Important caveat:** we scored *in-sample* here for teaching. The real pipeline uses **client-holdout** validation (`scripts/03_train_model.py`) so a client's pages never appear in both train and test. Re-run your comparison with a train/test split and see if the gap holds.

Write your experiment below.

**Completed below:** tree depths 2–4, feature substitution, a client holdout versus row-random comparison, a hand-rule comparison on each same test set, and a deliberately leaky control. Test-positive rates accompany ranking metrics.


In [7]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, average_precision_score
indices=np.arange(len(df))
group_train,group_test=next(GroupShuffleSplit(n_splits=1,test_size=.2,random_state=42).split(df,groups=df.client_id))
row_train,row_test=train_test_split(indices,test_size=.2,stratify=y,random_state=42)
assert set(df.iloc[group_train].client_id).isdisjoint(set(df.iloc[group_test].client_id))
records=[]
variants={'base':features,'drop_impressions_add_engagement':[f for f in features if f!='impressions_90d']+['engagement_rate']}
for split_name,(tr,te) in {'client_holdout':(group_train,group_test),'row_random':(row_train,row_test)}.items():
    for variant,cols in variants.items():
        for depth in [2,3,4]:
            model=make_pipeline(SimpleImputer(strategy='median'),DecisionTreeClassifier(max_depth=depth,class_weight='balanced',min_samples_leaf=30,random_state=42))
            model.fit(df.iloc[tr][cols],y[tr]);prob=model.predict_proba(df.iloc[te][cols])[:,1]
            records.append({'split':split_name,'features':variant,'depth':depth,'test_rows':len(te),'test_positive_rate':float(y[te].mean()),'roc_auc':float(roc_auc_score(y[te],prob)),'average_precision':float(average_precision_score(y[te],prob)),'precision_at_20':float(precision_at_k(prob,y[te],20)),'precision_at_50':float(precision_at_k(prob,y[te],50))})
    scores=df.iloc[te].hand_rule_score.to_numpy()
    records.append({'split':split_name,'features':'hand_rule','depth':0,'test_rows':len(te),'test_positive_rate':float(y[te].mean()),'roc_auc':float(roc_auc_score(y[te],scores)),'average_precision':float(average_precision_score(y[te],scores)),'precision_at_20':float(precision_at_k(scores,y[te],20)),'precision_at_50':float(precision_at_k(scores,y[te],50))})
comparison=pd.DataFrame(records)
print(comparison.round(4).to_string(index=False))
# Deliberately leaky control: trend_pct defines this proxy label; never use it in the real model.
cols=features+['trend_pct']
control=make_pipeline(SimpleImputer(strategy='median'),DecisionTreeClassifier(max_depth=4,random_state=42))
control.fit(df.iloc[group_train][cols],y[group_train]);leak_prob=control.predict_proba(df.iloc[group_test][cols])[:,1]
leak_auc=float(roc_auc_score(y[group_test],leak_prob))
print('Deliberately leaky held-out ROC AUC:',round(leak_auc,4))
print('This is rule reconstruction, not forecasting. Compare depth/feature variants cautiously: this is exploratory teaching validation, not sealed method selection.')
Path('work/outputs').mkdir(exist_ok=True)
Path('work/outputs/foundation_audit.json').write_text(json.dumps({'author':'Majhabul Hoque','seed':42,'rows':len(df),'proxy_label':'trend_direction equals down; contemporaneous, not future','comparison':records,'leaky_control_roc_auc':leak_auc,'group_client_overlap':0},indent=2)+'\n')


         split                        features  depth  test_rows  test_positive_rate  roc_auc  average_precision  precision_at_20  precision_at_50
client_holdout                            base      2       6163               0.511   0.5847             0.5591             0.70             0.72
client_holdout                            base      3       6163               0.511   0.5912             0.5666             0.70             0.66
client_holdout                            base      4       6163               0.511   0.5804             0.5672             0.55             0.62
client_holdout drop_impressions_add_engagement      2       6163               0.511   0.5470             0.5363             0.55             0.60
client_holdout drop_impressions_add_engagement      3       6163               0.511   0.5342             0.5354             0.50             0.64
client_holdout drop_impressions_add_engagement      4       6163               0.511   0.5202             0.5222      

### Save your work
**Colab:** *File → Save a copy in GitHub* (your submission) and *File → Save a copy in Drive*.

You now have the two core reflexes of applied ML: **discover before you model**, and **prefer a model you can read and can't fool**. That's the whole foundation the capstone builds on.